# 20. OVR ratings

Forecast, Phase 1, step 2e. Every NHL player and prospect gets an **EA NHL-style overall rating (OVR)**, with his projected WAR underneath. The season simulator will use the same scale, so it lives in a small module, `src/ratings.py`.

**The scale** (Ryan's decisions):

- EA-like: NHL regulars mostly 75-95, replacement level about 70, prospects and AHL-level players about 50-70, a floor of 40, the top player about 97-99.
- **One scale for all positions:** a 90 forward and a 90 defenseman are worth the same to a team.
- **Goalies** get their honest projections on the same scale, even if they cluster.

**The input is projected WAR/60** (`projections_2026-27.csv` after notebook 18's switch, and `prospects_2026-27.csv`). OVR measures talent, not minutes: a great player on a bad depth role still rates as great.

**Two steps:**

1. **WAR_std:** WAR/60 x the minutes of a full-time role at his position. That puts forwards, defensemen and goalies on the same footing: what would he be worth to a team in a full-time job?
2. **OVR:** WAR_std turned into a 40-99 number through a few fixed anchors (replacement = 70, median regular = 80, top player = 98), with straight lines between them.

Results stay private (`data/ratings_2026-27.csv`, gitignored). Nothing on the cards, app or README changes yet.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from ratings import war_std, ovr, ovr_exact, group_of, STD_MINUTES, ANCHOR_WAR, ANCHOR_OVR

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                      # where ratings_2026-27.csv is saved
NAMES = {"F": "Forwards", "D": "Defensemen", "G": "Goalies"}

proj = pd.read_csv(DATA / "projections_2026-27.csv")          # NHL players (notebook 18)
pros = pd.read_csv(DATA / "prospects_2026-27.csv")            # prospects (notebook 19)
skaters = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalies = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
sk25 = skaters[skaters["season"] == 2025].copy()              # 2025-26 (season = start year)
g25 = goalies[goalies["season"] == 2025].copy()
print(f"NHL players in the 2026-27 projection: {len(proj):,}; prospects: {len(pros):,}")
print(f"2025-26 seasons: {len(sk25):,} skaters, {len(g25):,} goalies, {sk25['team'].nunique()} teams")

NHL players in the 2026-27 projection: 1,328; prospects: 1,167
2025-26 seasons: 940 skaters, 98 goalies, 32 teams


## 2. A standard workload for each position

To compare positions on value, each one gets the minutes of a **full-time role**, measured from 2025-26:

- **Forwards:** each team's top 6 forwards by total ice time (the top two lines), average minutes per game, x 82 games.
- **Defensemen:** each team's top 4 by ice time (the top two pairs), same way.
- **Goalies:** each team's starter (most games played). A starter doesn't play all 82, so his workload is his **games x 60 minutes**.

Why not each player's own minutes? Because OVR should measure talent, not role. A 19-year-old stuck on the fourth line and a star on the top line are both rated as if they had a full-time job.

In [2]:
sk25["min_per_game"] = sk25["all_toi"] / sk25["games_played"]
rows = []
for g, n in [("F", 6), ("D", 4)]:
    # The n skaters with the most total ice time on each team (a traded player counts for his last team)
    top = sk25[sk25["pos_group"] == g].sort_values("all_toi", ascending=False).groupby("team").head(n)
    mpg = top["min_per_game"].mean()
    rows.append({"group": g, "full-time role": f"top {n} per team by ice time", "players": len(top),
                 "minutes per game": mpg, "games": 82, "median games played": top["games_played"].median(),
                 "standard minutes": mpg * 82})
starters = g25.sort_values("games_played", ascending=False).groupby("team").head(1)
rows.append({"group": "G", "full-time role": "starter (most games per team)", "players": len(starters),
             "minutes per game": 60, "games": starters["games_played"].mean(),
             "median games played": starters["games_played"].median(), "standard minutes": starters["games_played"].mean() * 60})
workload = pd.DataFrame(rows).set_index("group")
print(workload.round(2).to_string())
print(f"\nStarters' games ranged from {starters['games_played'].min()} to {starters['games_played'].max()}.")

# The frozen scale in src/ratings.py was set from these numbers. If the data ever changes, stop here
# instead of silently moving the scale.
for g in ["F", "D", "G"]:
    assert abs(workload.loc[g, "standard minutes"] - STD_MINUTES[g]) < 1, (g, workload.loc[g, "standard minutes"])
print("Frozen STD_MINUTES in ratings.py:", STD_MINUTES, "(match)")

                      full-time role  players  minutes per game  games  median games played  standard minutes
group                                                                                                        
F         top 6 per team by ice time      192             17.84  82.00                 79.0           1462.55
D         top 4 per team by ice time      128             21.35  82.00                 78.0           1751.07
G      starter (most games per team)       32             60.00  49.78                 50.5           2986.88

Starters' games ranged from 33 to 64.
Frozen STD_MINUTES in ratings.py: {'F': 1463, 'D': 1751, 'G': 2987} (match)


## 3. What each player is rated on

- **NHL players:** his 2026-27 projection (`new_proj_war_60`, notebook 18).
- **Prospect forwards:** his prospect projection (`proj_war_60`, notebook 19), the WAR/60 he'd produce **if he played NHL minutes**.
- **Prospect defensemen:** `d_rank_prior`. Notebook 19 gave every defenseman prospect the same projection (the position average), because no prior beat it on the test rookies. As a rating that would tie all 389 of them at one number, above the median NHL defenseman. The ranking prior spreads them out instead. Their `proj_war_60` (the simulator's input) stays the position average, so **for D prospects the rating and the simulator input differ**.
- **Prospect goalies:** a placeholder at replacement level (WAR/60 = 0, OVR 70). Their projection is the league-average goalie, which notebook 19 flagged as almost certainly too high for a rookie.

The column `rating_war_60` holds what the OVR uses and `rating_note` says which rule applied. `proj_war_60` is always the unchanged projection.

In [3]:
nhl = pd.DataFrame({"playerId": proj["playerId"], "name": proj["name"], "team": proj["team"], "position": proj["position"],
                    "group": group_of(proj["position"]), "age": proj["age"], "prospect": False,
                    "proj_war_60": proj["new_proj_war_60"], "rating_war_60": proj["new_proj_war_60"],
                    "rating_note": "NHL projection"})
pros_group = pros["group"]
pr = pd.DataFrame({"playerId": pros["playerId"], "name": pros["name"], "team": pros["draft_team"],
                   "position": pros["position"].replace({"LW": "L", "RW": "R"}), "group": pros_group,
                   "age": pros["age"], "prospect": True, "proj_war_60": pros["proj_war_60"],
                   "rating_war_60": np.select([pros_group == "D", pros_group == "G"], [pros["d_rank_prior"], 0.0], pros["proj_war_60"]),
                   "rating_note": np.select([pros_group == "D", pros_group == "G"],
                                            ["prospect D: d_rank_prior (ranking only)", "prospect G: replacement-level placeholder"],
                                            "prospect F: proj_war_60")})
assert not set(nhl["playerId"]) & set(pr["playerId"])          # nobody is in both lists
players = pd.concat([nhl, pr], ignore_index=True)
assert players["rating_war_60"].notna().all()

# 2025-26 minutes (skaters and goalies), for "regulars" and for the second WAR column
min25 = pd.concat([sk25.set_index("playerId")["all_toi"], g25.set_index("playerId")["toi"]])
gp25 = pd.concat([sk25.set_index("playerId")["games_played"], g25.set_index("playerId")["games_played"]])
players["minutes_2025_26"] = players["playerId"].map(min25).where(~players["prospect"])
players["war_std"] = war_std(players["rating_war_60"], players["group"])
players["regular"] = players["minutes_2025_26"] >= 500         # NHL regulars: 500+ minutes in 2025-26
print(players.groupby(["prospect", "group"]).size().rename("players").to_string())
print(f"\nNHL regulars (500+ minutes in 2025-26): {players['regular'].sum():,}")

prospect  group
False     D        407
          F        789
          G        132
True      D        389
          F        639
          G        139

NHL regulars (500+ minutes in 2025-26): 688


## 4. From WAR_std to OVR: the anchors

Four points, straight lines between them:

| WAR_std | OVR | Meaning |
|---|---|---|
| 0 | 70 | **replacement level**: the kind of player a team can call up from the AHL for free (WAR is measured above him) |
| median regular | 80 | the median NHL regular (500+ minutes in 2025-26, all positions) |
| top player | 98 | the top projected player for 2026-27 |
| below 0 | down to 40 | the replacement-to-median slope continues down to a floor of 40 |

Above the top player the median-to-top slope continues up to a cap of 99. The anchors are rounded to 0.05 WAR (the top one rounded down, so the top player lands at 98).

**Why the scale is frozen.** These numbers are now constants in `src/ratings.py`. The season simulator rates every simulated season with the same constants, so ratings can **rise and fall league-wide** in a multi-year sim: a strong generation can produce more 90s, a weak one fewer. Re-percentiling each season (always the same share of 90s) would hide that, so it's never done.

In [4]:
reg = players[players["regular"]]
nhl_only = players[~players["prospect"]]
median_reg, top = reg["war_std"].median(), nhl_only["war_std"].max()
top_name = nhl_only.loc[nhl_only["war_std"].idxmax(), "name"]
print(f"Median regular WAR_std: {median_reg:.3f}; top projected player: {top_name}, {top:.3f}")

# Proposed anchors from those two numbers
m = round(median_reg / 0.05) * 0.05                  # median, rounded to 0.05
t = np.floor(top / 0.05) * 0.05                      # top, rounded down to 0.05
floor_war = -3 * m                                   # 30 OVR points below 70 at the 0 -> median slope (10 per m)
cap_war = t + (t - m) / 18                           # one more OVR point at the median -> top slope
proposed = [round(floor_war, 2), 0.0, round(m, 2), round(t, 2), round(cap_war, 2)]
print(pd.DataFrame({"WAR_std": proposed, "OVR": [40, 70, 80, 98, 99],
                    "anchor": ["floor", "replacement", "median regular", "top player", "cap"]}).to_string(index=False))
print(f"Slopes: {10 / m:.1f} OVR per WAR between replacement and the median, {18 / (t - m):.1f} near the top")

assert proposed == ANCHOR_WAR and ANCHOR_OVR == [40, 70, 80, 98, 99], (proposed, ANCHOR_WAR)
print("Frozen anchors in ratings.py:", list(zip(ANCHOR_WAR, ANCHOR_OVR)), "(match)")

players["ovr_exact"] = ovr_exact(players["rating_war_60"], players["group"])
players["ovr"] = ovr(players["rating_war_60"], players["group"])
print(f"\n{top_name}: OVR {players.loc[players['war_std'].idxmax(), 'ovr']}")

Median regular WAR_std: 0.952; top projected player: Connor McDavid, 4.272
 WAR_std  OVR         anchor
   -2.85   40          floor
    0.00   70    replacement
    0.95   80 median regular
    4.25   98     top player
    4.43   99            cap
Slopes: 10.5 OVR per WAR between replacement and the median, 5.5 near the top
Frozen anchors in ratings.py: [(-2.85, 40), (0.0, 70), (0.95, 80), (4.25, 98), (4.43, 99)] (match)

Connor McDavid: OVR 98


## 5. Distribution checks

Before trusting the scale: where do NHL regulars land? Most of them should be 75-95.

In [5]:
reg = players[players["regular"]]
bins = list(range(40, 100, 5)) + [100]
hist = pd.cut(reg["ovr"].astype(float), bins, right=False).value_counts().sort_index()
print("NHL regulars by OVR (500+ minutes in 2025-26), each # = 4 players:")
for b, n in hist.items():
    print(f"  {b.left:>2}-{b.right - 1:<2} {n:>4}  {'#' * int(round(n / 4))}")

in_range = reg["ovr"].between(75, 95)
print(f"\nRegulars at 75-95: {in_range.mean():.1%} ({in_range.sum()} of {len(reg)})")
below = reg[reg["ovr"] < 75]
print(f"Below 75: {len(below)} ({len(below) / len(reg):.1%}): " + ", ".join(f"{NAMES[g]} {n}" for g, n in below["group"].value_counts().items()))
print(f"Above 95: {(reg['ovr'] > 95).sum()}")

summary = reg.groupby("group").agg(players=("ovr", "size"), median_war_std=("war_std", "median"),
                                    median_ovr=("ovr_exact", "median"), min_ovr=("ovr", "min"), max_ovr=("ovr", "max"),
                                    ovr_90_plus=("ovr", lambda x: int((x >= 90).sum())))
summary["share of group 90+"] = summary["ovr_90_plus"] / summary["players"]
print("\nBy position (regulars):")
print(summary.rename(index=NAMES).round(2).to_string())

NHL regulars by OVR (500+ minutes in 2025-26), each # = 4 players:
  40-44    0  
  45-49    0  
  50-54    0  
  55-59    2  
  60-64    9  ##
  65-69   52  #############
  70-74  123  ###############################
  75-79  134  ##################################
  80-84  213  #####################################################
  85-89  110  ############################
  90-94   39  ##########
  95-99    6  ##

Regulars at 75-95: 72.5% (499 of 688)
Below 75: 186 (27.0%): Defensemen 93, Forwards 93
Above 95: 3

By position (regulars):
            players  median_war_std  median_ovr  min_ovr  max_ovr  ovr_90_plus  share of group 90+
group                                                                                             
Defensemen      215            0.50       75.30       62       92            1                 0.0
Forwards        401            1.09       80.76       58       98           27                0.07
Goalies          72            2.33       87.54       81  

**Reading this:**

- **Defensemen rate lower than forwards on one scale.** The WAR model credits a defenseman less per full-time role than a forward, so the median defenseman sits below the median forward. That's what "one scale" means: a 90 defenseman really is worth as much as a 90 forward, there are just fewer of them.
- **Goalies cluster high.** Goalie results barely repeat from season to season (notebook 15), so their projections are pulled hard toward the league-average goalie, backups included. On a starter's workload the league-average goalie is well above replacement, so every NHL goalie rates well above 70. These are honest projections on the same scale; the cluster is the model saying "we can't tell goalies apart very well".

## 6. Hockey checks

The top of the scale, by position, the Red Wings, the prospects, and the bottom of the NHL.

`war_2025_minutes` (the second WAR column) is his projected WAR at **his own** 2025-26 workload: minutes per game x 82 for skaters, total 2025-26 minutes for goalies (minutes per game x 82 would mean starting every game). It's blank for prospects and for players who didn't play in 2025-26.

In [6]:
mpg25 = (players["playerId"].map(min25) / players["playerId"].map(gp25))
own_minutes = np.where(players["group"] == "G", players["minutes_2025_26"], mpg25 * 82)
players["war_2025_minutes"] = (players["proj_war_60"] * own_minutes / 60).where(players["minutes_2025_26"].notna() & ~players["prospect"])

SHOW = ["name", "team", "position", "age", "proj_war_60", "war_std", "ovr", "war_2025_minutes", "minutes_2025_26"]
nhl_only = players[~players["prospect"]]
print("Top 25 overall:")
print(nhl_only.nlargest(25, "ovr_exact")[SHOW].round(3).to_string(index=False))

Top 25 overall:
              name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
    Connor McDavid  EDM        C 30.0        0.175    4.272   98             5.506         1885.450
   Auston Matthews  TOR        C 29.0        0.171    4.179   98             4.873         1248.233
    Leon Draisaitl  EDM        C 31.0        0.162    3.941   96             4.766         1402.383
 Connor Hellebuyck  WPG        G 33.0        0.075    3.734   95             4.253         3401.450
    Logan Thompson  WSH        G 29.0        0.074    3.681   95             4.245         3445.050
     Cole Caufield  MTL        R 26.0        0.148    3.612   95             3.681         1472.867
    David Pastrnak  BOS        R 30.0        0.145    3.539   94             4.098         1590.483
  Nathan MacKinnon  COL        C 31.0        0.143    3.495   94             4.360         1780.950
   Igor Shesterkin  NYR        G 31.0        0.068    3.397   93             3.439  

In [7]:
for g in ["F", "D", "G"]:
    print(f"\nTop 10 {NAMES[g].lower()}:")
    print(nhl_only[nhl_only["group"] == g].nlargest(10, "ovr_exact")[SHOW].round(3).to_string(index=False))


Top 10 forwards:
             name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
   Connor McDavid  EDM        C 30.0        0.175    4.272   98             5.506         1885.450
  Auston Matthews  TOR        C 29.0        0.171    4.179   98             4.873         1248.233
   Leon Draisaitl  EDM        C 31.0        0.162    3.941   96             4.766         1402.383
    Cole Caufield  MTL        R 26.0        0.148    3.612   95             3.681         1472.867
   David Pastrnak  BOS        R 30.0        0.145    3.539   94             4.098         1590.483
 Nathan MacKinnon  COL        C 31.0        0.143    3.495   94             4.360         1780.950
Macklin Celebrini  SJS        C 20.0        0.139    3.380   93             4.038         1747.900
  Nikita Kucherov  TBL        R 33.0        0.138    3.358   93             3.827         1545.467
      Jack Hughes  NJD        C 25.0        0.131    3.191   92             3.812         1

In [8]:
det = nhl_only[nhl_only["team"] == "DET"].sort_values("ovr_exact", ascending=False)
print("Detroit Red Wings (last team in our data; players who didn't play in 2025-26 have no 2025-26 minutes):")
print(det[SHOW].round(3).to_string(index=False))

Detroit Red Wings (last team in our data; players who didn't play in 2025-26 have no 2025-26 minutes):
                   name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
         Alex DeBrincat  DET        R 29.0        0.119    2.906   91             3.013         1517.017
          Lucas Raymond  DET        L 24.0        0.114    2.790   90             2.940         1503.917
            John Gibson  DET        G 33.0        0.051    2.534   89             2.699         3181.233
           Dylan Larkin  DET        C 30.0        0.097    2.360   88             2.670         1493.850
        Sebastian Cossa  DET        G 24.0        0.047    2.325   87               NaN              NaN
     Michael Hutchinson  DET        G 36.0        0.039    1.951   85               NaN              NaN
             Cam Talbot  DET        G 39.0        0.036    1.800   85             1.054         1750.183
           Patrick Kane  DET        R 38.0        0.061  

**Prospects.** A prospect's rating means "if he played NHL minutes this season". Whether he does is Phase 2's call-up logic, not a rating discount. Defensemen are ranked by `d_rank_prior`; their `proj_war_60` is the shared position average.

In [9]:
pinfo = pros.set_index("playerId")[["draft_year", "pick", "league_2025_26", "gp_2025_26", "points_2025_26", "d_rank_prior"]]
pp = players[players["prospect"]].join(pinfo, on="playerId")
PSHOW = ["name", "position", "age", "team", "draft_year", "pick", "league_2025_26", "gp_2025_26", "points_2025_26",
         "proj_war_60", "d_rank_prior", "war_std", "ovr"]
print("Top 15 prospects by OVR:")
print(pp.nlargest(15, "ovr_exact")[PSHOW].round(3).to_string(index=False))
print("\nProspect OVR by group:")
print(pp.groupby("group")["ovr"].describe()[["count", "min", "25%", "50%", "75%", "max"]].rename(index=NAMES).to_string())
print("\nTop 5 defenseman prospects (by d_rank_prior):")
print(pp[pp["group"] == "D"].nlargest(5, "ovr_exact")[PSHOW].round(3).to_string(index=False))

Top 15 prospects by OVR:
               name position  age team  draft_year  pick league_2025_26  gp_2025_26  points_2025_26  proj_war_60  d_rank_prior  war_std  ovr
      Gavin McKenna        L 19.0  TOR        2026     1           NCAA        35.0            51.0        0.100           NaN    2.434   88
    Roman Kantserov        R 22.0  CHI        2023    44            KHL        63.0            64.0        0.094           NaN    2.288   87
    Andrew Cristall        L 21.0  WSH        2023    40            AHL        72.0            60.0        0.078           NaN    1.891   85
   Ethan Wyttenbach        L 19.0  CGY        2025   144           NCAA        40.0            59.0        0.070           NaN    1.709   84
         Max Plante        L 20.0  DET        2024    47           NCAA        40.0            52.0        0.067           NaN    1.640   84
Alexander Zharovsky        R 19.0  MTL        2025    34            KHL        59.0            42.0        0.066           NaN   

In [10]:
# Depth players: forwards who played 40+ games at under 12 minutes a night in 2025-26 (fourth-liners)
fourth = nhl_only[(nhl_only["group"] == "F") & (nhl_only["playerId"].map(gp25) >= 40) & (mpg25.reindex(nhl_only.index) < 12)]
print(f"Fourth-line forwards (40+ games, under 12 min/game in 2025-26): {len(fourth)}, "
      f"median OVR {fourth['ovr_exact'].median():.1f} (25th-75th percentile {fourth['ovr_exact'].quantile(0.25):.1f}-{fourth['ovr_exact'].quantile(0.75):.1f})")
print(fourth.sample(8, random_state=1).sort_values("ovr_exact", ascending=False)[SHOW].round(3).to_string(index=False))

# Third-pair defensemen: 40+ games at under 17 minutes a night
third = nhl_only[(nhl_only["group"] == "D") & (nhl_only["playerId"].map(gp25) >= 40) & (mpg25.reindex(nhl_only.index) < 17)]
print(f"\nThird-pair defensemen (40+ games, under 17 min/game): {len(third)}, median OVR {third['ovr_exact'].median():.1f}")

print("\nReplacement-level examples (regulars rated 70, most 2025-26 minutes first):")
print(nhl_only[nhl_only["regular"] & (nhl_only["ovr"] == 70)].nlargest(6, "minutes_2025_26")[SHOW].round(3).to_string(index=False))
print("\nLowest-rated regulars:")
print(nhl_only[nhl_only["regular"]].nsmallest(5, "ovr_exact")[SHOW].round(3).to_string(index=False))

Fourth-line forwards (40+ games, under 12 min/game in 2025-26): 77, median OVR 72.9 (25th-75th percentile 69.8-76.7)
            name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
      Max Sasson  VAN        C 26.0        0.039    0.949   80             0.623          773.450
  Jesper Boqvist  FLA        C 28.0        0.033    0.812   79             0.538          863.517
Hendrix Lapierre  WSH        C 24.0        0.025    0.607   76             0.300          653.717
  Jansen Harkins  ANA        C 29.0        0.020    0.477   75             0.257          421.983
  Ozzy Wiesblatt  NSH        C 24.0        0.001    0.036   70             0.021          407.283
    Kyle MacLean  NYI        C 27.0       -0.000   -0.001   70            -0.001          620.650
     Jeff Malott  LAK        L 30.0       -0.009   -0.215   68            -0.106          511.500
   Rodrigo Abols  PHI        C 31.0       -0.014   -0.333   66            -0.196          441.033



## 7. Save

`data/ratings_2026-27.csv`: one row per NHL player and prospect, with the projection, the WAR/60 the rating uses, WAR_std (**WAR per 82 games in a full-time role**; for goalies, per starter's season), OVR and the WAR at his 2025-26 minutes.

In [11]:
COLS = ["playerId", "name", "team", "position", "group", "age", "prospect", "proj_war_60", "rating_war_60", "rating_note",
        "war_std", "ovr", "war_2025_minutes"]
out = players[COLS].sort_values("ovr", ascending=False, kind="stable")
assert out["playerId"].is_unique and len(out) == len(proj) + len(pros)
out.to_csv(OUT / "ratings_2026-27.csv", index=False)
print(f"Saved ratings_2026-27.csv: {len(out):,} players ({(~out['prospect']).sum():,} NHL, {out['prospect'].sum():,} prospects)")

Saved ratings_2026-27.csv: 2,495 players (1,328 NHL, 1,167 prospects)


## Takeaways
- **The scale is frozen in `src/ratings.py`.**
  - **Standard workloads from 2025-26:** forwards 17.84 min/game (top 6 per team) = 1,463 minutes; defensemen 21.35 (top 4) = 1,751; goalies 49.78 starts (33 to 64) x 60 = 2,987.
  - **Anchors:** WAR_std 0 = 70 (replacement), 0.95 = 80 (median of 688 regulars, 0.952), 4.25 = 98 (McDavid, 4.272), with a floor of 40 at -2.85 and a cap of 99 at 4.43.
  - **What 1 WAR is worth:** 10.5 OVR points between replacement and the median, 5.5 near the top.
  - The simulator rates every simulated season with these constants, so ratings can rise and fall league-wide; nothing is re-percentiled.
- **Most regulars land 75-95:** 72.5% (499 of 688).
  - 186 are below 75, half of them defensemen (93 of 215 D).
  - Only 3 regulars are above 95: McDavid 98, Matthews 98, Draisaitl 96.
- **The top makes hockey sense.**
  - McDavid and Matthews 98, Draisaitl 96, Caufield 95, Pastrnak and MacKinnon 94, Celebrini 93 at 20.
  - Makar (92) is the top defenseman. Dahlin and Fox are 89, Werenski and Quinn Hughes 88.
  - The second WAR column shows workload matters: McDavid is 4.27 WAR in a standard role but 5.51 at his own 23 minutes a night.
- **Defensemen rate lower on one scale.** Median OVR is 75.3 for defensemen vs. 80.8 for forwards.
  - Only 1 of 215 regular defensemen is 90+, vs. 27 of 401 forwards.
  - The WAR model credits a defenseman less per full-time role, so "one scale" means fewer high-rated defensemen.
  - **Flag:** heavy-minute defensemen the model sees as replacement level rate 70 (Trouba, DeMelo, York, Mikkola: 1,350 to 1,850 minutes each). Chiarot is 62 on 1,709 minutes. Coaches clearly value them more than WAR does.
- **Goalies cluster high, honestly.**
  - Regular goalies run 81 to 95 (median 87.5), and 17 of 72 (24%) are 90+. Hellebuyck and Logan Thompson are 95.
  - Non-regulars rate high too: Cossa 87 (no NHL minutes in 2025-26), Hutchinson (36) 85, Talbot (39) 85, all above Seider (82).
  - The cause: goalie results barely repeat (notebook 15), so every goalie, backups included, is pulled hard toward the league-average goalie. On a starter's workload that goalie is well above replacement.
  - **A goalie prior by experience** (like notebook 18 for skaters) is a candidate first step of Phase 2.
- **Prospects are rated "if he plays NHL minutes".**
  - Forwards run 63 to 88 (median 74), above the 50-70 that was expected. Gavin McKenna is 88, equal to Larkin; then Kantserov 87 and Cristall 85.
  - **Defensemen use `d_rank_prior`** (68 to 80, median 71; Henry Mews 80), while their `proj_war_60` stays the position average. Phase 2 must know that for D prospects the rating and the simulator input differ.
  - **Goalie prospects** get a placeholder 70 (replacement level), flagged.
  - Readiness belongs in Phase 2's call-up logic, not in a rating discount.
- **The bottom of the NHL:**
  - Fourth-line forwards (77 with 40+ games under 12 min/game) have a median of 72.9 (25th-75th percentile 69.8-76.7), a little under the mid-70s guess.
  - Third-pair defensemen (49) have a median of 74.0.
  - The lowest regulars are Goodrow 58 and Ross Johnston 59.
- **Red Wings:** DeBrincat 91, Raymond 90, Gibson 89, Larkin 88, then goalies Cossa 87 and Talbot 85 (the goalie cluster), Kane 83 and Seider 82. Sandin-Pellikka and Finnie are 80, Edvinsson, Kasper and Danielson 78. Chiarot (62) and Hamonic (61) are lowest.
- **Next (Phase 2):** a goalie prior by experience, then minutes projections (who plays, how much, and which prospects get called up) and the season simulator, which imports `ratings.py`.